### Classification of Reuters News Wires


#### Workflow:
1. **Load and Prepare the Data**: We will use the Reuters dataset available via Keras/PyTorch utilities (or standard dataset tools) to get the index-mapped words, vectorize them into bag-of-words (one-hot) features, and convert them to PyTorch tensors.
2. **Build the Model**: Create a simple feedforward PyTorch `nn.Module` with Linear layers and ReLU activations.
3. **Train the Model**: Define our loss function (CrossEntropyLoss) and optimizer (Adam), and write a simple training loop.
4. **Evaluate**: Check how well our model performs on the test set.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from tensorflow.keras.datasets import reuters

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


#### Step 1: Data Preparation
Since Reuters dataset represents words as indices, we will represent each wire as a **Bag of Words** (multi-hot encoding vector of size 10,000, representing the most frequent words).

In [2]:
# Load Reuters dataset (keeping the top 10,000 most frequent words)
max_words = 10000
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=max_words, test_split=0.2)

# Helper function to convert index sequences to multi-hot vectors
def vectorize_sequences(sequences, dimension=max_words):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

# Vectorize training and testing data
X_train_vec = vectorize_sequences(x_train)
X_test_vec = vectorize_sequences(x_test)

# Convert NumPy arrays directly to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_vec, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_vec, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create PyTorch DataLoaders for efficient batch processing
batch_size = 64
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train shape: {X_train_tensor.shape}, Test shape: {X_test_tensor.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")

2110848/2110848 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train shape: torch.Size([8982, 10000]), Test shape: torch.Size([2246, 10000])
Number of classes: 46


#### Step 2: Define the Neural Network Architecture
We design a simple network using `nn.Module` with two hidden layers of 64 units each, and an output layer of 46 units (corresponding to the 46 topics).

In [3]:
class ReutersClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(ReutersClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out) # CrossEntropyLoss in PyTorch expects raw logits, so no Softmax layer is needed here
        return out

# Instantiate the model
input_dimension = max_words
hidden_dimension = 64
num_classes = 46

model = ReutersClassifier(input_dimension, hidden_dimension, num_classes)
print(model)

ReutersClassifier(
  (fc1): Linear(in_features=10000, out_features=64, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=64, out_features=46, bias=True)
)


#### Step 3: Train the Network
We define our Loss Function (`CrossEntropyLoss`) and our optimizer (`Adam`). Then, we write a clean loop to train the network over several epochs.

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for batch_x, batch_y in train_loader:
        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        # Backward pass & Optimize
        loss.backward()
        optimizer.step()

        # Compute metrics
        running_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == batch_y).sum().item()
        total_samples += batch_y.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions / total_samples

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.4f}")

Epoch [1/10] - Loss: 1.9911 - Accuracy: 0.5915
Epoch [2/10] - Loss: 1.0150 - Accuracy: 0.7673
Epoch [3/10] - Loss: 0.6067 - Accuracy: 0.8663
Epoch [4/10] - Loss: 0.3756 - Accuracy: 0.9169
Epoch [5/10] - Loss: 0.2658 - Accuracy: 0.9361
Epoch [6/10] - Loss: 0.2089 - Accuracy: 0.9482
Epoch [7/10] - Loss: 0.1864 - Accuracy: 0.9487
Epoch [8/10] - Loss: 0.1674 - Accuracy: 0.9501
Epoch [9/10] - Loss: 0.1580 - Accuracy: 0.9530
Epoch [10/10] - Loss: 0.1437 - Accuracy: 0.9528


#### Step 4: Evaluate the Model on Test Data

In [5]:
model.eval()
test_loss = 0.0
correct_predictions = 0
total_samples = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        test_loss += loss.item() * batch_x.size(0)
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == batch_y).sum().item()
        total_samples += batch_y.size(0)

final_loss = test_loss / total_samples
final_acc = correct_predictions / total_samples

print(f"Test Loss: {final_loss:.4f}")
print(f"Test Accuracy: {final_acc * 100:.2f}%")

Test Loss: 1.0956
Test Accuracy: 79.03%
